In [1]:
# Import required packages
from meijer import Meijer
import pandas as pd
from datetime import datetime
import time

# Try to import tabulate for better table formatting
try:
    from tabulate import tabulate
    TABULATE_AVAILABLE = True
    print("✅ Tabulate available for enhanced table formatting")
except ImportError:
    TABULATE_AVAILABLE = False
    print("📝 Install tabulate for better tables: pip install tabulate")

# Initialize the Meijer client
print("🚀 Initializing Meijer client...")
client = Meijer()

# Check authentication status
if client.auth_status.name == "AUTHENTICATED":
    print("✅ Successfully authenticated!")
    print(f"🔐 Auth method: {client.auth_status}")
else:
    print("❌ Authentication failed. Please check your credentials.")
    print("   Ensure you have auth.txt or ~/.config/meijer.txt configured")


TypeError: non-default argument 'point_cost' follows default argument

In [2]:
# Get current shopping list
print("📋 Getting current shopping list...")
current_items = client.list.get()

print(f"Found {len(current_items)} items in shopping list")

if current_items:
    # Display current items in a table
    current_data = []
    
    for i, item in enumerate(current_items, 1):
        current_data.append({
            'Order': i,
            'Item': item.name[:40],  # Truncate long names
            'Quantity': item.quantity,
            'Notes': item.notes[:30] if item.notes else 'None',
            'Status': '✅ Done' if item.checked else '⏳ Pending',
            'UPC': item.upc or 'N/A'
        })
    
    df_current = pd.DataFrame(current_data)
    print("\n📊 Current Shopping List (Before Defrag):")
    display(df_current)
    
    print(f"\n💡 Notice: Items are in the order they were added, not organized by store layout")
else:
    print("📝 Your shopping list is empty!")
    print("\n🛒 Add some items first:")
    print("   • Use the Meijer app to add items")
    print("   • Or use: client.list.add_item_with_details('UPC', description='Item Name')")


📋 Getting current shopping list...


NameError: name 'client' is not defined

In [3]:
# Add sample items if the list is empty or small
if len(current_items) < 3:
    print("🛒 Adding sample items for demonstration...")
    
    sample_items = [
        {'description': 'Milk', 'upc': '0001234567890'},
        {'description': 'Bread', 'upc': '0001234567891'},
        {'description': 'Cereal', 'upc': '0001234567892'},
        {'description': 'Oreos', 'upc': '0001234567893'},
        {'description': 'Ground Turkey', 'upc': '0001234567894'}
    ]
    
    added_count = 0
    for item in sample_items:
        success = client.list.add_item_with_details(
            upc=item['upc'],
            description=item['description'],
            quantity=1
        )
        if success:
            added_count += 1
            print(f"  ✅ Added: {item['description']}")
        else:
            print(f"  ❌ Failed to add: {item['description']}")
        
        time.sleep(0.5)  # Be nice to the API
    
    print(f"\n📊 Added {added_count} sample items")
    
    # Refresh the list
    current_items = client.list.get()
    print(f"📋 Updated list now has {len(current_items)} items")
else:
    print("📋 Using existing items in your shopping list")


NameError: name 'current_items' is not defined

In [4]:
# Run the defrag process
print("🔧 Starting shopping list defrag process...")
print("⏳ This may take a moment as we search for each item's location")
print("-" * 60)

# Run defrag (you can optionally specify a store_id)
defrag_success = client.list.defrag()

print("-" * 60)
if defrag_success:
    print("🎉 Defrag completed successfully!")
else:
    print("❌ Defrag encountered some issues")


🔧 Starting shopping list defrag process...
⏳ This may take a moment as we search for each item's location
------------------------------------------------------------


NameError: name 'client' is not defined

In [5]:
# Get the defragged shopping list
print("📋 Getting defragged shopping list...")
defragged_items = client.list.get()

print(f"Found {len(defragged_items)} items in defragged list")

if defragged_items:
    # Display defragged items in a detailed table
    defragged_data = []
    
    for i, item in enumerate(defragged_items, 1):
        # Extract detailed information from enhanced notes
        aisle_info = 'Unknown'
        matched_product = 'No match'
        brand_info = 'N/A'
        price_info = 'N/A'
        confidence = 'Unknown'
        
        if item.notes:
            # Parse enhanced notes for detailed information
            if 'Aisle:' in item.notes:
                try:
                    aisle_part = item.notes.split('Aisle:')[1].split('|')[0].strip()
                    aisle_info = aisle_part
                except:
                    pass
            
            if 'Matched:' in item.notes:
                try:
                    match_part = item.notes.split('Matched:')[1].split('|')[0].strip()
                    matched_product = match_part[:40] + '...' if len(match_part) > 40 else match_part
                except:
                    pass
            
            if 'Brand:' in item.notes:
                try:
                    brand_part = item.notes.split('Brand:')[1].split('|')[0].strip()
                    brand_info = brand_part
                except:
                    pass
            
            if 'Price:' in item.notes:
                try:
                    price_part = item.notes.split('Price:')[1].split('|')[0].strip()
                    price_info = price_part
                except:
                    pass
            
            if 'Match Confidence:' in item.notes:
                try:
                    conf_part = item.notes.split('Match Confidence:')[1].split('|')[0].strip()
                    confidence = conf_part
                except:
                    pass
        
        defragged_data.append({
            'Order': i,
            'Original Item': item.name[:25] + '...' if len(item.name) > 25 else item.name,
            'Closest Match': matched_product,
            'Brand': brand_info,
            'Price': price_info,
            'Aisle': aisle_info,
            'Confidence': confidence,
            'Quantity': item.quantity,
            'Status': '✅ Done' if item.checked else '⏳ Pending'
        })
    
    df_defragged = pd.DataFrame(defragged_data)
    print("\n📊 Defragged Shopping List (Organized by Aisle with Match Details):")
    display(df_defragged)
    
    # Show aisle distribution
    aisle_counts = {}
    for item_data in defragged_data:
        aisle = item_data['Aisle']
        aisle_counts[aisle] = aisle_counts.get(aisle, 0) + 1
    
    print("\n🏪 Items by Aisle:")
    for aisle, count in sorted(aisle_counts.items(), key=lambda x: (x[0] == 'Unknown', x[0])):
        print(f"  📍 Aisle {aisle}: {count} item(s)")
    
    # Show confidence distribution
    confidence_counts = {}
    for item_data in defragged_data:
        conf = item_data['Confidence']
        confidence_counts[conf] = confidence_counts.get(conf, 0) + 1
    
    print("\n🎯 Match Confidence Distribution:")
    for conf, count in confidence_counts.items():
        print(f"  {conf}: {count} item(s)")
        
else:
    print("📝 No items found in the defragged list")


📋 Getting defragged shopping list...


NameError: name 'client' is not defined

In [6]:
# Show example of enhanced notes
if defragged_items:
    print("📝 Example of Enhanced Notes (showing first 3 items):")
    print("=" * 80)
    
    for i, item in enumerate(defragged_items[:3], 1):
        print(f"\n{i}. {item.name}")
        print(f"   Quantity: {item.quantity}")
        if item.notes:
            print(f"   Enhanced Notes:")
            # Split notes by | and format nicely
            note_parts = item.notes.split(' | ')
            for part in note_parts:
                print(f"      • {part}")
        else:
            print(f"   Notes: None")
    
    print("\n💡 Benefits of Enhanced Notes:")
    print("  ✅ Know exactly what product was matched")
    print("  ✅ See brand and price information")
    print("  ✅ Understand match confidence level")
    print("  ✅ Have complete location details")
    print("  ✅ Preserve your original notes")
else:
    print("📝 No items to show enhanced notes for")


NameError: name 'defragged_items' is not defined

In [7]:
# Analyze the defrag benefits
print("🎯 Defrag Analysis & Benefits:")
print("=" * 50)

if defragged_items:
    # Count items with location data
    items_with_aisle = 0
    items_with_section = 0
    items_with_match = 0
    items_with_price = 0
    items_with_brand = 0
    
    for item in defragged_items:
        if item.notes:
            if 'Aisle:' in item.notes:
                items_with_aisle += 1
            if 'Section:' in item.notes:
                items_with_section += 1
            if 'Matched:' in item.notes:
                items_with_match += 1
            if 'Price:' in item.notes:
                items_with_price += 1
            if 'Brand:' in item.notes:
                items_with_brand += 1
    
    benefits_data = {
        'Metric': [
            'Total Items',
            'Items with Aisle Info',
            'Items with Section Info',
            'Items with Product Match',
            'Items with Price Info',
            'Items with Brand Info',
            'Location Coverage',
            'Organization Status'
        ],
        'Value': [
            len(defragged_items),
            items_with_aisle,
            items_with_section,
            items_with_match,
            items_with_price,
            items_with_brand,
            f"{(items_with_aisle/len(defragged_items)*100):.1f}%",
            "✅ Organized by Aisle"
        ]
    }
    
    benefits_df = pd.DataFrame(benefits_data)
    display(benefits_df)
    
    print("\n🛒 Shopping Efficiency Improvements:")
    print("  ✅ Items organized by ascending aisle number")
    print("  ✅ Location information added to notes")
    print("  ✅ Logical shopping route through the store")
    print("  ✅ Reduced backtracking and missed items")
    print("  ✅ Faster, more efficient shopping experience")
    
    print("\n🔍 Enhanced Product Information:")
    print("  ✅ Know exactly what product was matched")
    print("  ✅ Verify brand and price before shopping")
    print("  ✅ Understand match confidence levels")
    print("  ✅ Catch incorrect matches (e.g., Milk vs Milk of Magnesia)")
    print("  ✅ Complete product details in item notes")
    
    if items_with_aisle < len(defragged_items):
        missing_count = len(defragged_items) - items_with_aisle
        print(f"\n💡 Note: {missing_count} item(s) don't have aisle information")
        print("   This could be because:")
        print("   • Item is not available in-store")
        print("   • Search didn't find a matching product")
        print("   • Item location data is not available")
        
    if items_with_match < len(defragged_items):
        no_match_count = len(defragged_items) - items_with_match
        print(f"\n⚠️  Warning: {no_match_count} item(s) couldn't be matched to products")
        print("   Consider:")
        print("   • Checking spelling of item names")
        print("   • Using more specific product names")
        print("   • Verifying items are available at Meijer")
else:
    print("❌ No items to analyze")


🎯 Defrag Analysis & Benefits:


NameError: name 'defragged_items' is not defined

In [8]:
# Final summary
print("🎊 Shopping List Defrag Demo Complete!")
print("=" * 50)

final_items = client.list.get()
if final_items:
    print(f"📋 Your list now has {len(final_items)} organized items")
    print("🏪 Items are sorted by aisle for efficient shopping")
    print("📱 Use this organized list in the Meijer app or mobile site")
    
    print("\n🚶‍♀️ Shopping Route Preview:")
    for i, item in enumerate(final_items[:5], 1):  # Show first 5 items
        aisle = 'Unknown'
        if item.notes and 'Aisle:' in item.notes:
            try:
                aisle = item.notes.split('Aisle:')[1].split('|')[0].strip()
            except:
                pass
        print(f"  {i}. {item.name} → Aisle {aisle}")
    
    if len(final_items) > 5:
        print(f"  ... and {len(final_items) - 5} more items")
        
    print("\n💡 Pro Tip: Your shopping list is now optimized for a logical")
    print("   route through the store, saving you time and steps!")
else:
    print("📝 No items in final list")

print("\n✨ Happy organized shopping! ✨")


🎊 Shopping List Defrag Demo Complete!


NameError: name 'client' is not defined